# Instruction-Tuning with Large Language Models (LLMs)

Instruction-based fine-tuning, commonly known as *Instruction GPT*, is a training approach where language models are optimized to understand and follow human instructions accurately. The objective is to help the model generate meaningful and task-specific responses based on user prompts.

In instruction-tuning, the dataset is highly important because it provides structured examples containing instructions, optional context, and expected outputs. By learning from these examples, the model becomes capable of handling a wide range of real-world tasks more effectively.

Many Instruction GPT systems are further improved using human feedback to refine response quality and alignment. However, this project focuses only on the instruction fine-tuning process and does not include reinforcement learning or human feedback optimization.

The instruction and context are typically merged into a single input sequence so the model can better understand the task and generate an appropriate response.

## Key Components

- **Instruction**  
  A task description or command that tells the model what action to perform.

- **Context**  
  Additional background information or supporting details required to complete the task correctly.

- **Combined Input**  
  The instruction and context are combined into one sequence before being passed to the model during training or inference.


### **Import Libraries**

In [3]:
import torch
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
from datasets import load_dataset 
from transformers import AutoTokenizer, AutoModelForCausalLM, Pipeline

### **Load Dataset**

In [21]:
SEED = 42

In [11]:
dataset = load_dataset("sahil2801/CodeAlpaca-20k")

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\datasets--sahil2801--CodeAlpaca-20k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 20022/20022 [00:00<00:00, 101691.12 examples/s]


In [12]:
dataset.column_names

{'train': ['output', 'instruction', 'input']}

In [15]:
dataset['train'][1000]

{'output': 's = "Hello world" \ns = s[::-1] \nprint(s)',
 'instruction': 'Reverse the string given in the input',
 'input': 'Hello world'}

In [16]:
dataset = dataset['train']
dataset[1000]

{'output': 's = "Hello world" \ns = s[::-1] \nprint(s)',
 'instruction': 'Reverse the string given in the input',
 'input': 'Hello world'}

**Filter the Dataset Which Don't Have Any Input**

In [20]:
dataset = dataset.filter(lambda example: example['input'] == '')
dataset[0:2]

Filter: 100%|██████████| 9764/9764 [00:00<00:00, 67260.41 examples/s]


{'output': ['arr = [2, 4, 6, 8, 10]',
  'Height of triangle = opposite side length * sin (angle) / side length'],
 'instruction': ['Create an array of length 5 which contains all even numbers between 1 and 10.',
  'Formulate an equation to calculate the height of a triangle given the angle, side lengths and opposite side length.'],
 'input': ['', '']}

In [24]:
### shuffle the dataset
dataset = dataset.shuffle(seed=SEED)

In [25]:
dataset

Dataset({
    features: ['output', 'instruction', 'input'],
    num_rows: 9764
})

In [35]:
dataset['instruction'][0]

'Name the most important benefit of using a database system.'

**Split Dataset**

In [26]:
train_test_split = dataset.train_test_split(test_size=0.2,seed=SEED)
dataset_train = train_test_split['train']
dataset_test = train_test_split['test']
train_test_split

DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 7811
    })
    test: Dataset({
        features: ['output', 'instruction', 'input'],
        num_rows: 1953
    })
})

### **Model & Tokenizer Implementation**

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [31]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m", padding_side = 'left')

In [32]:
tokenizer

GPT2Tokenizer(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, padding_side='left', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})

In [34]:
tokenizer.eos_token

'</s>'

In [33]:
model = AutoModelForCausalLM.from_pretrained("facebook/opt-350m")
model.to(device)

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 22825.50it/s]


OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=409

### Data Pre-Processing 

##### Formatting the prompt

In [36]:
def fromatting_prompt_with_response (mydataset):
    response = []
    
    for i in range(len(mydataset)):
        text = (
            f"### Instruction: \n{mydataset['instruction'][i]}"
            f"\n\n ### Response: \n{mydataset['output'][i]}"
        )
        
        response.append(text)
    return response

In [38]:
def formatting_prompt_with_no_response(mydataset):
    
    response = []
    
    for i in range(len(mydataset)):
        text = (
            f"### Instruction: \n{mydataset['instruction'][i]}"
            f"\n\n### Response:\n"
        )
        
        response.append(text)
    return response